In [3]:
import pandas as pd
import numpy as np
import json
import joblib

df = pd.read_csv("sample_data/sri_lanka_flood_data.csv")

features = [
    "elevation_m",
    "distance_to_river_m",
    "rainfall_7d_mm",
    "monthly_rainfall_mm",
    "drainage_index",
    "population_density_per_km2",
    "built_up_percent",
    "infrastructure_score"
]

# Keep only clean feature rows and encode target
df = df.dropna(subset=features + ["flood_occurrence_current_event"]).copy()
X = df[features]
y = (df["flood_occurrence_current_event"].astype(str).str.strip().str.lower() == "yes").astype(int)

# Save feature min/max/median so Streamlit sliders match dataset bounds automatically
feature_stats = {
    col: {
        "min": float(X[col].min()),
        "max": float(X[col].max()),
        "median": float(X[col].median())
    }
    for col in features
}
with open("feature_stats.json", "w") as f:
    json.dump(feature_stats, f, indent=2)

print("Dataset shape:", X.shape)
print("Target distribution:\n", y.value_counts(normalize=True))

Dataset shape: (25000, 8)
Target distribution:
 flood_occurrence_current_event
0    0.90112
1    0.09888
Name: proportion, dtype: float64


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 1. BAGGING (Parallel trees on bootstrap samples to reduce variance)
bagging_model = RandomForestClassifier(
    n_estimators=150, max_depth=12, random_state=42, n_jobs=-1
)

# 2. BOOSTING (Sequential trees correcting prior errors to reduce bias)
boosting_model = XGBClassifier(
    n_estimators=150, learning_rate=0.08, max_depth=6,
    eval_metric="logloss", random_state=42, n_jobs=-1
)

# 3. STACKING (Combines diverse base learners using a Logistic Regression meta-learner)
base_learners = [
    ("rf", RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)),
    ("xgb", XGBClassifier(n_estimators=100, learning_rate=0.08, max_depth=5, eval_metric="logloss", random_state=42, n_jobs=-1)),
    ("et", ExtraTreesClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))
]
stacking_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=make_pipeline(StandardScaler(), LogisticRegression()),
    cv=5,
    n_jobs=-1
)

models = {
    "Bagging (Random Forest)": bagging_model,
    "Boosting (XGBoost)": boosting_model,
    "Stacking (RF + XGB + ET)": stacking_model
}

metrics_summary = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics_summary[name] = {
        "Accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
        "Precision": round(float(precision_score(y_test, y_pred)), 4),
        "Recall": round(float(recall_score(y_test, y_pred)), 4),
        "F1_Score": round(float(f1_score(y_test, y_pred)), 4),
        "ROC_AUC": round(float(roc_auc_score(y_test, y_prob)), 4),
        "Confusion_Matrix": confusion_matrix(y_test, y_pred).tolist(),
        "Feature_Importances": (
            dict(zip(features, [round(float(v), 4) for v in model.feature_importances_]))
            if hasattr(model, "feature_importances_") else {}
        )
    }

print(json.dumps(metrics_summary, indent=2))

{
  "Bagging (Random Forest)": {
    "Accuracy": 0.9134,
    "Precision": 0.6034,
    "Recall": 0.3603,
    "F1_Score": 0.4512,
    "ROC_AUC": 0.9586,
    "Confusion_Matrix": [
      [
        4389,
        117
      ],
      [
        316,
        178
      ]
    ],
    "Feature_Importances": {
      "elevation_m": 0.1042,
      "distance_to_river_m": 0.1078,
      "rainfall_7d_mm": 0.2362,
      "monthly_rainfall_mm": 0.3038,
      "drainage_index": 0.0726,
      "population_density_per_km2": 0.0601,
      "built_up_percent": 0.0557,
      "infrastructure_score": 0.0596
    }
  },
  "Boosting (XGBoost)": {
    "Accuracy": 0.916,
    "Precision": 0.5898,
    "Recall": 0.4919,
    "F1_Score": 0.5364,
    "ROC_AUC": 0.9576,
    "Confusion_Matrix": [
      [
        4337,
        169
      ],
      [
        251,
        243
      ]
    ],
    "Feature_Importances": {
      "elevation_m": 0.1236,
      "distance_to_river_m": 0.0967,
      "rainfall_7d_mm": 0.2926,
      "monthly_rainfall

In [5]:
from google.colab import files

joblib.dump(bagging_model, "bagging_model.joblib", compress=3)
joblib.dump(boosting_model, "boosting_model.joblib", compress=3)
joblib.dump(stacking_model, "stacking_model.joblib", compress=3)

with open("metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

# Print exact versions to put in requirements.txt
import sklearn, xgboost
print(f"scikit-learn=={sklearn.__version__}")
print(f"xgboost=={xgboost.__version__}")

# Download artifacts for GitHub
for filename in [
    "bagging_model.joblib",
    "boosting_model.joblib",
    "stacking_model.joblib",
    "metrics.json",
    "feature_stats.json"
]:
    files.download(filename)

scikit-learn==1.6.1
xgboost==3.4.1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>